google collab depedencies

In [1]:
!pip -q install bertopic
!pip -q install sastrawi
!pip -q install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 34.9 MB/s eta 0:00:00


In [2]:
!git clone -q -b gavriel-thesis https://github.com/ranslemus/topic_modeling_KBMI4.git
%cd topic_modeling_KBMI4

/content/topic_modeling_KBMI4


In [3]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import plotly.express as px

from transformers import AutoTokenizer, AutoModel
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from tqdm.auto import tqdm
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from hdbscan.validity import validity_index

# for linux
from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN

# for windows
# import umap as UMAP
# import hdbscan as HDBSCAN

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

if device.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

Device : cuda
GPU : Tesla T4


In [42]:
df = pd.read_csv("data/preprocessed_data.csv")
df = df[df['year'] == 2025]
df.head()

,reviewId,bank,score,year,text
0,e17751da-bf2e-4a8f-a5a8-334206bb93ca,BCAMOBILE_REVIEWS,1,2025,ribet banget nih apk sumpah dikir verif dikit ...
1,3481f1d1-a22f-445f-ae2f-ed8c2c9dca53,BCAMOBILE_REVIEWS,2,2025,kenapa qris enggak bisa di pakai ya daritadi l...
2,af69ec6d-cc97-404e-b86a-e5f5b5f1f711,BCAMOBILE_REVIEWS,1,2025,aplikasi nya sampah kenapa tiba tiba keluar te...
3,32d28ac6-c538-4749-969f-8af060915d96,BCAMOBILE_REVIEWS,3,2025,bagus
4,f99259b7-0d45-4422-b667-6c4fd521638b,BCAMOBILE_REVIEWS,1,2025,tidak ada solusi ketika ada kendala di persuli...


In [43]:
df["word_count"] = df["text"].astype(str).str.split().apply(len)
df = df[df["word_count"] >= 5].reset_index(drop=True)
print(f"Total documents setelah filter: {len(df):,}")

Total documents setelah filter: 52,055


In [44]:
documents = df["text"].astype(str).tolist()

print(f"Total documents : {len(documents):,}")

Total documents : 52,055


# IndoBERT

In [45]:
MODEL_NAME = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModel.from_pretrained(MODEL_NAME)

model.to(device)

model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(50000, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [46]:
def mean_pooling(model_output, attention_mask):

    token_embeddings = model_output.last_hidden_state

    input_mask_expanded = (
        attention_mask
        .unsqueeze(-1)
        .expand(token_embeddings.size())
        .float()
    )

    return torch.sum(
        token_embeddings * input_mask_expanded,
        dim=1
    ) / torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )

In [47]:
def encode_documents(
    documents,
    batch_size=32,
    max_length=128
):

    embeddings = []

    with torch.no_grad():

        for i in tqdm(
            range(0, len(documents), batch_size)
        ):

            batch = documents[
                i:i+batch_size
            ]

            encoded_input = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            encoded_input = {
                k: v.to(device)
                for k, v in encoded_input.items()
            }

            model_output = model(**encoded_input)

            sentence_embeddings = mean_pooling(
                model_output,
                encoded_input["attention_mask"]
            )

            sentence_embeddings = (
                sentence_embeddings
                .cpu()
                .numpy()
            )

            embeddings.append(sentence_embeddings)

    return np.vstack(embeddings)

In [48]:
embeddings = encode_documents(
    documents,
    batch_size=32,
    max_length=128
)

  0%|          | 0/1627 [00:00<?, ?it/s]

In [49]:
print(embeddings.shape)

(52055, 768)


In [50]:
norms = np.linalg.norm(embeddings, axis=1)

print("Minimum Norm :", norms.min())
print("Maximum Norm :", norms.max())
print("Average Norm :", norms.mean())
print("Std Norm :", norms.std())

Minimum Norm : 12.880096
Maximum Norm : 25.153244
Average Norm : 17.674175
Std Norm : 1.4323688


In [51]:
print("NaN :", np.isnan(embeddings).sum())
print("Inf :", np.isinf(embeddings).sum())

NaN : 0
Inf : 0


In [52]:
# np.save(
#     "indobert_embeddings.npy",
#     embeddings
# )

# BERTopic

In [53]:
# embeddings = np.load("indobert_embeddings.npy")

print("Embedding Shape :", embeddings.shape)

Embedding Shape : (52055, 768)


In [54]:
sastrawi_stopwords = StopWordRemoverFactory().get_stop_words()

# extra_particles = ["banget", "terus", "padahal", "sih", "aja", "saja", "dong", "deh", "ya", "kok", "biar", "gitu", "nih", "loh", "mau", "sudah", "belum"]
# sastrawi_stopwords_extended = sastrawi_stopwords + extra_particles

vectorizer_model = CountVectorizer(
  ngram_range=(1,2),
  stop_words=sastrawi_stopwords,
  token_pattern=r"(?u)\b[^\d\W]+\b",
  min_df=2,
  )

baseline UMAP for testing purpose

In [55]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    metric="cosine",
    min_dist=0.0,
    random_state=42
)

baseline HDBSCAN

In [56]:
hdbscan_model = HDBSCAN(
    min_cluster_size=100,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="leaf",
    prediction_data=True
)

In [57]:
topic_model = BERTopic(
    embedding_model=None,
    calculate_probabilities=False,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=True
)

In [58]:
topics, probabilities = topic_model.fit_transform(
    documents,
    embeddings
)

2026-08-11 09:00:45,288 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-11 09:00:47,450 - BERTopic - Dimensionality - Completed ✓
2026-08-11 09:00:47,452 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-11 09:00:47,910 - BERTopic - Cluster - Completed ✓
2026-08-11 09:00:47,924 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-11 09:00:49,586 - BERTopic - Representation - Completed ✓


# Evaluation

Basic Statistics

In [59]:
topic_info = topic_model.get_topic_info()

topic_info.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,39933,-1_enggak_aplikasi_mau_nya,"[enggak, aplikasi, mau, nya, terus, padahal, m...",[kenapa buka livin susah banget ya padahal buk...
1,0,1009,0_malam_jam_pemeliharaan_maintenance,"[malam, jam, pemeliharaan, maintenance, jam ma...",[kebiasaan banget kalo pemeliharaan jam malam ...
2,1,909,1_benar_salah_pasword_password,"[benar, salah, pasword, password, username, lo...",[kenapa ini ya saya enggak bisa masuk akun bri...
3,2,648,2_enggak buka_update_update malah_buka,"[enggak buka, update, update malah, buka, mala...",[di update malah enggak bisa di buka kenapa in...
4,3,608,3_kok_buka_enggak buka_kok enggak,"[kok, buka, enggak buka, kok enggak, enggak, d...","[growing saya kok enggak bisa di buka, kenapa ..."
5,4,603,4_saldo_gagal saldo_gagal_qris,"[saldo, gagal saldo, gagal, qris, kepotong, be...","[transaksi gagal tapi saldo berkurang, transak..."
6,5,586,5_perbaiki_diperbaiki_mohon_segera,"[perbaiki, diperbaiki, mohon, segera, nya, moh...","[sering gangguan tolong segera di perbaiki, ma..."
7,6,537,6_buka_enggak buka_dibuka_aplikasi,"[buka, enggak buka, dibuka, aplikasi, enggak, ...","[kenapa enggak bisa di buka aplikasi ya, kenap..."
8,7,438,7_nasabah_prioritas_abrik_nasabah prioritas,"[nasabah, prioritas, abrik, nasabah prioritas,...",[nasabah prioritas saja di obrak abrik apalagi...
9,8,429,8_susah_daftar_mau daftar_mau,"[susah, daftar, mau daftar, mau, susah banget,...","[ribet mau daftar saja susah, susah banget mau..."


In [60]:
num_topics = len(topic_info) - 1

outlier_count = (np.array(topics) == -1).sum()

outlier_percentage = (
    outlier_count / len(topics)
) * 100

print(f"Topics              : {num_topics}")
print(f"Outliers            : {outlier_count:,}")
print(f"Outlier Percentage  : {outlier_percentage:.2f}%")

Topics              : 40
Outliers            : 39,933
Outlier Percentage  : 76.71%


Topic Size

In [61]:
topic_info[["Topic","Count"]]

,Topic,Count
0,-1,39933
1,0,1009
2,1,909
3,2,648
4,3,608
5,4,603
6,5,586
7,6,537
8,7,438
9,8,429


Top Words

In [62]:
top_10_topics = topic_model.get_topic_info()
top_10_topics = top_10_topics[top_10_topics.Topic != -1].nlargest(10, "Count")

for _, row in top_10_topics.iterrows():
    topic_id = row['Topic']
    doc_count = row['Count']

    print("=" * 80)
    print(f"TOPIC {topic_id} | JUMLAH DOKUMEN: {doc_count}")
    print("=" * 80)

    # Menampilkan word-score pair bawaan BERTopic (c-TF-IDF scores)
    words_with_scores = topic_model.get_topic(topic_id)
    for word, score in words_with_scores:
        print(f"  - {word:<20} : {score:.4f}")
    print()

TOPIC 0 | JUMLAH DOKUMEN: 1009
  - malam                : 0.0989
  - jam                  : 0.0988
  - pemeliharaan         : 0.0423
  - maintenance          : 0.0405
  - jam malam            : 0.0397
  - tiap                 : 0.0350
  - tengah malam         : 0.0315
  - tengah               : 0.0314
  - tiap malam           : 0.0276
  - transaksi            : 0.0272

TOPIC 1 | JUMLAH DOKUMEN: 909
  - benar                : 0.0504
  - salah                : 0.0422
  - pasword              : 0.0395
  - password             : 0.0374
  - username             : 0.0367
  - login                : 0.0322
  - padahal              : 0.0297
  - sandi                : 0.0246
  - akun                 : 0.0184
  - padahal benar        : 0.0170

TOPIC 2 | JUMLAH DOKUMEN: 648
  - enggak buka          : 0.0695
  - update               : 0.0691
  - update malah         : 0.0584
  - buka                 : 0.0569
  - malah enggak         : 0.0553
  - enggak               : 0.0503
  - malah              

Representative Reviews

In [63]:
# Ambil info topik dan urutkan berdasarkan jumlah dokumen terbesar (kecuali outlier -1)
topic_info = topic_model.get_topic_info()
top_10_topics = topic_info[topic_info.Topic != -1].nlargest(10, "Count")["Topic"].tolist()

print("=== TOP 10 TOPIK PALING REPRESENTATIF ===")

for topic_id in top_10_topics:
    # Ambil ukuran klaster asli
    cluster_size = topic_info.loc[topic_info.Topic == topic_id, "Count"].values[0]

    # Ambil kata kunci utama topik untuk mempermudah pembacaan aspek
    keywords = ", ".join([w for w, _ in topic_model.get_topic(topic_id)[:5]])

    # Ambil dokumen yang secara matematis paling dekat dengan centroid klaster (Bawaan BERTopic)
    rep_docs = topic_model.get_representative_docs(topic_id)

    print("\n" + "=" * 120)
    print(f"TOPIC {topic_id} | CLUSTER SIZE: {cluster_size}")
    print(f"KEYWORDS : {keywords}")
    print("=" * 120)

    # BERTopic menyimpan maksimum 3 representative docs per topik secara default
    for i, doc in enumerate(rep_docs, 1):
        print(f"{i}. {doc}")

=== TOP 10 TOPIK PALING REPRESENTATIF ===

TOPIC 0 | CLUSTER SIZE: 1009
KEYWORDS : malam, jam, pemeliharaan, maintenance, jam malam
1. kebiasaan banget kalo pemeliharaan jam malam semua juga ada keperluan jangan apa apa pemeliharaan mulu waktu nya cukup lama coba jangan jam malam untuk pemeliharaan nya kalo seperti ini semua akan susah tidak semua pegang cash tolong di perbaiki jam jam pemeliharaan nya min
2. kadang kadang suka gagal transaksi di tengah malam di jam malam
3. kenapa ya setiap jam 12 malam lebih tidak bisa transaksi tolong penjelasanya

TOPIC 1 | CLUSTER SIZE: 909
KEYWORDS : benar, salah, pasword, password, username
1. kenapa ini ya saya enggak bisa masuk akun brimo padahal username dan password sudah benar tapi tetap saja enggak masuk keterangan username dan password salah jelas-jelas sudah benar parah nih
2. kenapa ya enggak bisa login padahal username dan password sudah benar tapi tetap tidak bisa
3. kok enggak bisa login padahal username dan password sudah benar

TOP

silhoutte score

In [64]:
from sklearn.metrics import silhouette_score

mask = np.array(topics) != -1

silhouette = silhouette_score(
    topic_model.umap_model.embedding_[mask],
    np.array(topics)[mask]
)

print(f"Silhouette Score : {silhouette:.4f}")

Silhouette Score : 0.5352


In [65]:
from itertools import chain

top_n = 10
topic_words = []

for topic in topic_info.Topic:
    if topic == -1:
        continue

    words = [
        word
        for word, score in topic_model.get_topic(topic)[:top_n]
    ]
    topic_words.append(words)

unique_words = len(
    set(chain.from_iterable(topic_words))
)

total_words = len(topic_words) * top_n
topic_diversity = unique_words / total_words

print(f"Topic Diversity : {topic_diversity:.4f}")

Topic Diversity : 0.7300


NPMI

In [66]:
analyzer = topic_model.vectorizer_model.build_analyzer()

In [67]:
doc.split()

['enggak', 'bisa', 'login', 'padahal', 'jaringan', 'bagus']

In [68]:
tokenized_docs = [
    analyzer(doc)
    for doc in documents
]

In [69]:
from gensim.corpora import Dictionary

dictionary = Dictionary(tokenized_docs)
topic_words = []

for topic in topic_info.Topic:

    if topic == -1:
        continue

    words = []

    for word, score in topic_model.get_topic(topic):
        if word in dictionary.token2id:
            words.append(word)
    # Need at least 2 words for coherence
    if len(words) >= 2:
        topic_words.append(words)

In [70]:
# sanity check
print(f"Valid Topics : {len(topic_words)}")

print()

print(topic_words[:3])

Valid Topics : 40

[['malam', 'jam', 'pemeliharaan', 'maintenance', 'jam malam', 'tiap', 'tengah malam', 'tengah', 'tiap malam', 'transaksi'], ['benar', 'salah', 'pasword', 'password', 'username', 'login', 'padahal', 'sandi', 'akun', 'padahal benar'], ['enggak buka', 'update', 'update malah', 'buka', 'malah enggak', 'enggak', 'malah', 'dibuka', 'enggak dibuka', 'habis update']]


In [71]:
from gensim.models.coherencemodel import CoherenceModel

coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence="c_npmi"
)

npmi = coherence_model.get_coherence()
print(f"NPMI : {npmi:.4f}")

NPMI : 0.1360


DBCV

In [72]:
mask = np.array(topics) != -1
X = topic_model.umap_model.embedding_[mask].astype(np.float64)
labels = np.array(topics)[mask]

dbcv_score = validity_index(X, labels)
print(f"DBCV : {dbcv_score:.4f}")

DBCV : 0.2072


In [73]:
import pandas as pd
from scipy.stats import chi2_contingency

df["topic"] = topics

# 1. Baseline: proporsi tiap bank di keseluruhan korpus
baseline = df["bank"].value_counts(normalize=True) * 100
print("Proporsi bank di keseluruhan korpus (baseline):")
print(baseline.round(2))
print()

# 2. Proporsi tiap bank DI DALAM tiap topik
crosstab = pd.crosstab(df["topic"], df["bank"], normalize="index") * 100
crosstab = crosstab.round(2)

# 3. Hitung "lift" = proporsi di topik / proporsi baseline
#    >1 artinya over-represented di topik itu, <1 artinya under-represented
lift = crosstab.copy()
for bank in baseline.index:
    lift[bank] = crosstab[bank] / baseline[bank]

# 4. Tandai topik yang "njomplang" (deviasi lift > 1.5x atau < 0.5x dari baseline)
def flag_imbalance(row):
    return any(row > 1.5) or any(row < 0.5)

lift["is_imbalanced"] = lift[baseline.index].apply(flag_imbalance, axis=1)

# gabung count per topik biar gampang liat mana yang topik "besar" (bukan cuma noise kecil)
topic_sizes = df[df["topic"] != -1]["topic"].value_counts()
lift["topic_size"] = lift.index.map(topic_sizes)

result = lift[lift.index != -1].sort_values("is_imbalanced", ascending=False)
print(result[list(baseline.index) + ["is_imbalanced", "topic_size"]])

Proporsi bank di keseluruhan korpus (baseline):
bank
BRIMO_REVIEWS            35.60
LIVIN_MANDIRI_REVIEWS    27.78
WONDR_BNI_REVIEWS        21.80
BCAMOBILE_REVIEWS        14.82
Name: proportion, dtype: float64

bank   BRIMO_REVIEWS  LIVIN_MANDIRI_REVIEWS  WONDR_BNI_REVIEWS  \
topic                                                            
0           0.320269               2.076009           0.932274   
26          1.254107               1.435355           0.382177   
1           1.879194               0.732434           0.449161   
21          0.632672               0.972859           1.921897   
22          1.507232               0.449898           0.334462   
23          1.897736               0.574430           0.634515   
24          1.515099               0.970700           0.541379   
25          0.500631               0.579109           2.742224   
28          0.381233               0.591346           0.655620   
17          0.021913               0.028074           0.000000 